# Bab 13 — Introducing automatic optimization: let’s build a deep learning framework

**Mahasiswa:** Tarannia Raisha Putri · 101032300231 · TK4702 · Deep Learning

**Referensi utama:** Andrew W. Trask, *Grokking Deep Learning*, Manning (2019), Bab 13, halaman cetak 231–263 (PDF 252–284, indeks PDF mulai 1). Subbab spesifik dicantumkan pada setiap tahap.

**Tujuan pembelajaran:** Membangun Tensor/autograd, memperbaiki penggunaan tensor berulang, mengimplementasikan layer/optimizer, dan melatih contoh buku.

## Ringkasan

Bab ini mengubah perhitungan gradien manual menjadi framework kecil. Tensor menyimpan nilai dan graph komputasi, backward menjumlahkan jalur turunan, SGD memperbarui parameter, dan layer menyusun operasi menjadi model. Semua operasi tetap berbasis NumPy.

## Konsep, intuisi, dan notasi

Jika $z=a+b$, upstream gradient $g$ dikirim ke keduanya. Jika $z=ab$, gradiennya $gb$ dan $ga$. Untuk $Z=AB$, $G_A=G_ZB^T$ dan $G_B=A^TG_Z$. Satu tensor yang dipakai di banyak jalur menerima **jumlah** kontribusi, bukan nilai terakhir.

`sum(axis)` mengurangi dimensi, backward mengembalikan sumbu itu dan menyiarkan gradien; `expand` menyalin nilai, backward menjumlahkan salinannya. `transpose` dibalik dengan transpose. `index_select` harus menjumlahkan gradien untuk indeks berulang. Cross-entropy atas logits $(B,C)$ memberi $(P-Y)/B$ jika loss mean.

Framework bukan pengganti chain rule: ia menyimpan operasi forward agar dapat menjalankan aturan yang sama secara otomatis. Kernel ini tidak mendukung higher-order derivatives; broadcasting umum matmul juga tidak didukung. `Tensor.data` dan `Tensor.grad` adalah ndarray float, parameter mempunyai `autograd=True`, indeks tetap bilangan bulat di antarmuka indexing.

## Persiapan dan cara membaca

Jalankan **Restart Kernel → Run All**. Semua penjelasan merupakan parafrasa orisinal; angka hasil berada pada output sel yang benar-benar dijalankan. Label **reproduksi** berarti algoritme/contoh buku ditulis ulang; **koreksi/penyesuaian** menunjukkan perubahan; **tambahan** menunjukkan eksperimen atau interpretasi praktikum. `GDL_MODE=quick` mengurangi latihan berat; `full` memilih anggaran buku, dengan perubahan yang tetap dicatat. Keberhasilan eksekusi tidak membuktikan kesamaan akurasi dengan buku.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Jalankan dari root repository atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = os.environ.get('GDL_MODE', 'quick')
assert MODE in {'quick', 'full'}
FULL = MODE == 'full'
np.random.seed(1)
np.set_printoptions(precision=6, suppress=True)
METRICS = {}
def record(**values):
    METRICS.update(values)
def finite(*arrays):
    assert all(np.isfinite(np.asarray(a)).all() for a in arrays), 'NaN/inf terdeteksi'
def softmax(z):
    z = np.asarray(z)
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z)))
print('Mode:', MODE, '| NumPy:', np.__version__)

Mode: quick | NumPy: 2.5.4


<a id="b13-tensor"></a>

## Reproduksi bertahap: tensor dan autograd pertama

**Sumber:** cetak 232–236; PDF 253–257.

Kelas pertama hanya membungkus array dan addition. Kelas kedua menunjukkan graph penjumlahan serta bug overwrite gradien. Mempertahankan versi yang sengaja salah membantu menjelaskan alasan pengembangan berikutnya.

In [2]:
class ValueTensor:
    def __init__(self,data): self.data=np.array(data)
    def __add__(self,other): return ValueTensor(self.data+other.data)
x=ValueTensor([1,2,3,4,5]); print('Tensor:',x.data,'x+x:',(x+x).data)
class NaiveTensor(ValueTensor):
    def __init__(self,data,parents=()): super().__init__(data); self.parents=parents; self.grad=None
    def __add__(self,other): return NaiveTensor(self.data+other.data,(self,other))
    def backward(self,grad):
        self.grad=np.array(grad)
        for parent in self.parents: parent.backward(grad)
a=NaiveTensor([1,2,3,4,5]); b=NaiveTensor([2]*5); c=NaiveTensor([5,4,3,2,1])
f=(a+b)+(b+c); f.backward(np.ones(5))
print('Bug awal: grad b=',b.grad,'padahal seharusnya 2 pada setiap komponen.')
assert np.array_equal(b.grad,np.ones(5))

Tensor: [1 2 3 4 5] x+x: [ 2  4  6  8 10]
Bug awal: grad b= [1. 1. 1. 1. 1.] padahal seharusnya 2 pada setiap komponen.


<a id="b13-graph"></a>

## Reproduksi operasi dan penyesuaian graph untuk multiuse

**Sumber:** cetak 237–245; PDF 258–266.

Buku memakai ID acak dan penghitung anak agar gradien lengkap sebelum dikirim ke parent. Implementasi berikut memakai **traversal topologis** dengan identitas objek: konsep akumulasi sama, tetapi mekanisme pengurutan sengaja direfaktor. Urutan maju mengunjungi parent dahulu; backward membalik urutan itu. Parameter yang dipakai ulang diproses sekali setelah semua kontribusi masuk.

Seluruh fungsi Tensor dicantumkan, termasuk negation, subtraction, multiplication, sum, expand, transpose, matrix multiplication, sigmoid, tanh, indexing, dan cross-entropy. Metode lanjutan diperkenalkan lagi melalui demonstrasi di bawah. `backward()` membersihkan gradien graph aktif; penggabungan loss harus dilakukan sebelum memanggil backward, bukan lewat beberapa backward yang diharapkan mengakumulasi otomatis.

In [3]:
"""Implementasi pedagogis NumPy berdasarkan operasi Bab 13 (Trask, Manning).

Penyesuaian struktur: traversal topologis menggantikan children-count rekursif.
Gradien adalah ndarray, loss mean konsisten, indeks berulang memakai scatter-add.
Tidak mendukung higher-order gradients atau seluruh broadcasting NumPy matmul.
Kode ini juga ditampilkan lengkap dan bertahap di notebook Bab 13.
"""
import numpy as np


def unbroadcast(gradient, shape):
    """Undo broadcasting: setiap penggunaan elemen menjumlahkan kontribusi gradien."""
    while gradient.ndim > len(shape):
        gradient = gradient.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1:
            gradient = gradient.sum(axis=axis, keepdims=True)
    return gradient.reshape(shape)


class Tensor:
    def __init__(self, data, autograd=False, parents=(), op='', backward=None):
        self.data = np.array(data, dtype=float)
        self.autograd = autograd
        self.grad = np.zeros_like(self.data)
        self.parents = tuple(parents)
        self.op = op
        self._backward = backward or (lambda g: None)

    def __repr__(self):
        return f'Tensor({self.data!r}, op={self.op!r})'

    @staticmethod
    def wrap(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, op, backward):
        return Tensor(data, any(p.autograd for p in parents), parents, op, backward)

    def _accumulate(self, gradient):
        if self.autograd:
            self.grad += unbroadcast(np.asarray(gradient), self.data.shape)

    def __add__(self, other):
        other = Tensor.wrap(other)
        def backward(g):
            self._accumulate(g)
            other._accumulate(g)
        return self._result(self.data+other.data,(self,other),'add',backward)

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data,(self,),'neg',lambda g:self._accumulate(-g))

    def __sub__(self, other):
        return self + (-Tensor.wrap(other))

    def __mul__(self, other):
        other = Tensor.wrap(other)
        def backward(g):
            self._accumulate(g*other.data)
            other._accumulate(g*self.data)
        return self._result(self.data*other.data,(self,other),'mul',backward)

    __rmul__ = __mul__

    def mm(self, other):
        other = Tensor.wrap(other)
        assert self.data.ndim == other.data.ndim == 2
        def backward(g):
            self._accumulate(g@other.data.T)
            other._accumulate(self.data.T@g)
        return self._result(self.data@other.data,(self,other),'mm',backward)

    __matmul__ = mm

    def transpose(self):
        assert self.data.ndim == 2
        return self._result(self.data.T,(self,),'transpose',lambda g:self._accumulate(g.T))

    def sum(self, dim=None):
        def backward(g):
            restored = np.expand_dims(g, dim) if dim is not None else g
            self._accumulate(np.ones_like(self.data)*restored)
        return self._result(self.data.sum(axis=dim),(self,),'sum',backward)

    def mean(self, dim=None):
        count = self.data.size if dim is None else self.data.shape[dim]
        return self.sum(dim)*(1/count)

    def expand(self, dim, copies):
        expanded = np.repeat(np.expand_dims(self.data,dim),copies,axis=dim)
        return self._result(expanded,(self,),'expand',lambda g:self._accumulate(g.sum(axis=dim)))

    def sigmoid(self):
        value = np.exp(-np.logaddexp(0,-self.data))
        return self._result(value,(self,),'sigmoid',lambda g:self._accumulate(g*value*(1-value)))

    def tanh(self):
        value = np.tanh(self.data)
        return self._result(value,(self,),'tanh',lambda g:self._accumulate(g*(1-value**2)))

    def index_select(self, indices):
        indices = indices.data if isinstance(indices,Tensor) else np.asarray(indices)
        assert np.all(indices == indices.astype(int))
        indices = indices.astype(int)
        def backward(g):
            if self.autograd:
                np.add.at(self.grad,indices,g)
        return self._result(self.data[indices],(self,),'index_select',backward)

    def cross_entropy(self, target):
        target = target.data if isinstance(target,Tensor) else np.asarray(target)
        target = target.astype(int).ravel()
        assert self.data.ndim == 2 and len(target) == len(self.data)
        z = self.data-self.data.max(axis=1,keepdims=True)
        log_probs = z-np.log(np.exp(z).sum(axis=1,keepdims=True))
        prob = np.exp(log_probs)
        value = -log_probs[np.arange(len(target)),target].mean()
        def backward(g):
            delta = prob.copy()
            delta[np.arange(len(target)),target] -= 1
            self._accumulate(g*delta/len(target))
        return self._result(value,(self,),'cross_entropy',backward)

    def detach(self):
        return Tensor(self.data.copy(), self.autograd)

    def backward(self, gradient=None):
        # Setiap node diproses setelah semua anak dalam graph aktif selesai.
        order=[]; visited=set()
        def visit(node):
            if id(node) in visited:
                return
            visited.add(id(node))
            for parent in node.parents:
                visit(parent)
            order.append(node)
        visit(self)
        # Satu panggilan backward = satu graph; jangan membawa gradien batch lama.
        for node in order:
            node.grad = np.zeros_like(node.data)
        if gradient is None:
            gradient=np.ones_like(self.data)
        if isinstance(gradient,Tensor):
            gradient=gradient.data
        self.grad=np.asarray(gradient,dtype=float)
        assert self.grad.shape == self.data.shape
        for node in reversed(order):
            if node.autograd:
                node._backward(node.grad)
        for node in order:
            assert np.isfinite(node.grad).all(), f'Gradien tidak finite: {node.op}'

In [4]:
a=Tensor([1,2,3,4,5],True); b=Tensor([2]*5,True); c=Tensor([5,4,3,2,1],True)
f=(a+b)+(b+c); f.backward()
print('Multiuse grad b:',b.grad); np.testing.assert_allclose(b.grad,2)
# Reused intermediate, bukan hanya reused leaf.
x=Tensor([2.],True); q=x*x; f=q+q; f.backward(); np.testing.assert_allclose(x.grad,[8.])
x=Tensor([[1,2,3],[4,5,6]],True)
print('sum(0):',x.sum(0).data,'sum(1):',x.sum(1).data,'expand:',x.expand(2,4).data.shape)
print('neg:',(-x).data,'sub:',(x-x).data,'transpose:',x.transpose().data)
x.expand(2,4).sum().backward(); np.testing.assert_allclose(x.grad,4)
# Tambahan finite difference komposisi operasi.
a=Tensor([[.2,-.3],[.1,.4]],True); b=Tensor([[.5],[-.2]],True)
def loss_fn(): return ((a.mm(b).tanh()-Tensor([[.1],[.2]]))*(a.mm(b).tanh()-Tensor([[.1],[.2]]))).sum()
loss_fn().backward(); expected=a.grad.copy(); numerical=np.zeros_like(a.data); eps=1e-6
for idx in np.ndindex(a.data.shape):
    old=a.data[idx]; a.data[idx]=old+eps; plus=loss_fn().data.item()
    a.data[idx]=old-eps; minus=loss_fn().data.item(); a.data[idx]=old
    numerical[idx]=(plus-minus)/(2*eps)
autograd_error=float(np.max(np.abs(expected-numerical))); assert autograd_error<1e-7
print('Gradient check graph:',autograd_error)

Multiuse grad b: [2. 2. 2. 2. 2.]
sum(0): [5. 7. 9.] sum(1): [ 6. 15.] expand: (2, 3, 4)
neg: [[-1. -2. -3.]
 [-4. -5. -6.]] sub: [[0. 0. 0.]
 [0. 0. 0.]] transpose: [[1. 4.]
 [2. 5.]
 [3. 6.]]
Gradient check graph: 6.3307276088053754e-12


<a id="b13-training"></a>

## Reproduksi: manual versus autograd dan SGD

**Sumber:** cetak 246–248; PDF 267–269.

Data empat pasangan dan jaringan 2→3→1 mengikuti buku. Satu update manual dan otomatis harus sama untuk squared-error sum. Nama MSE pada buku sebenarnya menjumlahkan squared error; framework di sini memakai mean secara eksplisit. Perubahan skala dicatat, sehingga learning rate tidak dibandingkan tanpa melihat reduksi loss.

In [5]:
np.random.seed(0)
X=np.array([[0,0],[0,1],[1,0],[1,1]],float); Y=np.array([[0],[1],[0],[1]],float)
A=np.random.rand(2,3); B=np.random.rand(3,1)
h=X@A; pred=h@B; delta=2*(pred-Y)
gB=h.T@delta; gA=X.T@(delta@B.T)
a=Tensor(A,True); b=Tensor(B,True); prediction=Tensor(X).mm(a).mm(b)
loss=((prediction-Tensor(Y))*(prediction-Tensor(Y))).sum(); loss.backward()
np.testing.assert_allclose(a.grad,gA); np.testing.assert_allclose(b.grad,gB)
print('Gradien manual dan autograd sama. Loss:',loss.data)
manual_A=A-.1*gA; manual_B=B-.1*gB
a.data-=.1*a.grad; b.data-=.1*b.grad
np.testing.assert_allclose(a.data,manual_A); np.testing.assert_allclose(b.data,manual_B)

Gradien manual dan autograd sama. Loss: 5.066439994622395


In [6]:
class SGD:
    def __init__(self, parameters, alpha=.1):
        self.parameters=list(parameters)
        self.alpha=alpha

    def zero(self):
        for p in self.parameters:
            p.grad.fill(0)

    def step(self, zero=True):
        for p in self.parameters:
            p.data -= self.alpha*p.grad
            assert np.isfinite(p.data).all()
        if zero:
            self.zero()


class Layer:
    def get_parameters(self):
        return []

    def __call__(self,*args,**kwargs):
        return self.forward(*args,**kwargs)


class Linear(Layer):
    def __init__(self,n_inputs,n_outputs,bias=True):
        self.weight=Tensor(np.random.randn(n_inputs,n_outputs)*np.sqrt(2/n_inputs),True)
        self.bias=Tensor(np.zeros(n_outputs),True) if bias else None

    def forward(self,x):
        out=x.mm(self.weight)
        return out+self.bias.expand(0,len(x.data)) if self.bias is not None else out

    def get_parameters(self):
        return [self.weight]+([] if self.bias is None else [self.bias])


class Sequential(Layer):
    def __init__(self,layers=None):
        self.layers=list(layers or [])

    def add(self,layer):
        self.layers.append(layer)

    def forward(self,x):
        for layer in self.layers:
            x=layer(x)
        return x

    def get_parameters(self):
        return [p for layer in self.layers for p in layer.get_parameters()]


class MSELoss(Layer):
    def forward(self,pred,target):
        difference=pred-target
        return (difference*difference).mean()


class Tanh(Layer):
    def forward(self,x):
        return x.tanh()


class Sigmoid(Layer):
    def forward(self,x):
        return x.sigmoid()


class Embedding(Layer):
    def __init__(self,vocab_size,dim):
        self.weight=Tensor((np.random.rand(vocab_size,dim)-.5)/dim,True)

    def forward(self,indices):
        return self.weight.index_select(indices)

    def get_parameters(self):
        return [self.weight]


class CrossEntropyLoss(Layer):
    def forward(self,pred,target):
        return pred.cross_entropy(target)


class RNNCell(Layer):
    def __init__(self,n_inputs,n_hidden,n_output,activation='sigmoid'):
        self.n_hidden=n_hidden
        self.w_ih=Linear(n_inputs,n_hidden)
        self.w_hh=Linear(n_hidden,n_hidden)
        self.w_ho=Linear(n_hidden,n_output)
        self.activation={'sigmoid':Sigmoid,'tanh':Tanh}[activation]()

    def forward(self,x,hidden):
        hidden=self.activation(self.w_ih(x)+self.w_hh(hidden))
        return self.w_ho(hidden),hidden

    def init_hidden(self,batch_size=1):
        return Tensor(np.zeros((batch_size,self.n_hidden)),True)

    def get_parameters(self):
        return self.w_ih.get_parameters()+self.w_hh.get_parameters()+self.w_ho.get_parameters()

**Mengapa layer terpisah?** `Linear` mengelompokkan bobot dan bias, `Sequential` menghubungkan layer, dan loss adalah operasi yang dapat diturunkan. `SGD` hanya mengubah data parameter dari gradien yang sudah tersedia. `Embedding` memilih baris tanpa membuat matriks one-hot penuh; `RNNCell` memakai ulang bobot pada setiap langkah waktu. Ini pembungkusan operasi sebelumnya, bukan framework eksternal.

<a id="b13-layers"></a>

## Reproduksi: layers, sequential, loss, nonlinearities

**Sumber:** cetak 249–254; PDF 270–275.

Tiga tahap dijalankan: tensor langsung dengan optimizer, susunan layer linear, lalu tanh+sigmoid. Target adalah bit kedua, **bukan XOR**. Mean loss dan 10 langkah dipakai supaya tahap-tahap dapat dibandingkan secara jelas; jumlah langkah latihan tambahan dapat dinaikkan tanpa mengubah operasi.

In [7]:
np.random.seed(0)
a=Tensor(np.random.rand(2,3),True); b=Tensor(np.random.rand(3,1),True)
optimizer=SGD([a,b],.1)
for step in range(10):
    pred=Tensor(X).mm(a).mm(b); loss=MSELoss()(pred,Tensor(Y)); loss.backward(); optimizer.step()
print('Tensor + SGD loss akhir:',loss.data)
for label,model in [('linear',Sequential([Linear(2,3),Linear(3,1)])),('nonlinear',Sequential([Linear(2,3),Tanh(),Linear(3,1),Sigmoid()]))]:
    optimizer=SGD(model.get_parameters(),.05); history=[]
    for step in range(10):
        pred=model(Tensor(X)); loss=MSELoss()(pred,Tensor(Y)); history.append(loss.data.item())
        loss.backward(); optimizer.step()
    print(label,'loss:',history)
    assert history[-1]<history[0]

Tensor + SGD loss akhir: 0.06877397604126269
linear loss: [2.106338987999531, 0.07047810475913231, 0.030255297022815916, 0.025662519395719258, 0.02252208774977918, 0.019830054483584443, 0.017497562580892266, 0.015470442104211295, 0.01370398461475842, 0.012160733397606987]
nonlinear loss: [0.45053867890060423, 0.4481152308626509, 0.4457051423983475, 0.44330866698470506, 0.4409260423703927, 0.43855749014213186, 0.43620321536667583, 0.43386340631042286, 0.4315382342379528, 0.4292278532900204]


<a id="b13-embedding"></a>

## Reproduksi: embedding, indexing, cross-entropy

**Sumber:** cetak 255–259; PDF 276–280.

Indeks berulang `[1,1,2]` menguji akumulasi gradien. Cross-entropy menerima logits dan indeks kelas, menghitung log-softmax stabil, dan memasukkan pembagi batch di backward. Pada listing, forward mean tetapi backward tidak membagi batch; koreksi ini mengubah skala update.

In [8]:
x=Tensor(np.eye(5),True); selected=x.index_select([1,1,2]); selected.sum().backward()
print('Gradien baris berulang:',x.grad)
np.testing.assert_allclose(x.grad[1],2); np.testing.assert_allclose(x.grad[2],1)
indices=np.array([1,2,1,2]); target=np.array([0,1,0,1])
for kind in ['mse','cross_entropy']:
    np.random.seed(0)
    model=Sequential([Embedding(5,3),Linear(3,1 if kind=='mse' else 4)])
    optim=SGD(model.get_parameters(),.1); history=[]
    for step in range(10):
        pred=model(indices)
        loss=MSELoss()(pred,Tensor(target[:,None])) if kind=='mse' else CrossEntropyLoss()(pred,target)
        history.append(loss.data.item()); loss.backward(); optim.step()
    print(kind,'loss:',history); assert history[-1]<history[0]
# Gradien cross-entropy batch juga diperiksa numerik.
z=Tensor([[.2,-.1,.5],[.1,.4,-.2]],True); target=np.array([2,1]); z.cross_entropy(target).backward(); grad=z.grad.copy()
for idx in np.ndindex(z.data.shape):
    old=z.data[idx]; z.data[idx]=old+eps; plus=z.cross_entropy(target).data.item()
    z.data[idx]=old-eps; minus=z.cross_entropy(target).data.item(); z.data[idx]=old
    assert abs((plus-minus)/(2*eps)-grad[idx])<1e-7

Gradien baris berulang: [[0. 0. 0. 0. 0.]
 [2. 2. 2. 2. 2.]
 [1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
mse loss: [0.46994652560634237, 0.3463545454116552, 0.26798556818492203, 0.2157873466412054, 0.17896329063539024, 0.15132476303556114, 0.1293263470710458, 0.11095652842760031, 0.09509425305398522, 0.08112984618088923]
cross_entropy loss: [1.2932801078786662, 1.2515012456975851, 1.212479286270217, 1.1758913807302922, 1.1414535604771077, 1.1089146891823607, 1.0780516083914455, 1.0486652341488105, 1.0205774143476651, 0.9936283968586255]


<a id="b13-rnn"></a>

## Reproduksi: recurrent neural network layer

**Sumber:** cetak 260–263; PDF 281–284.

RNN memakai tiga Linear: input→hidden, hidden→hidden, dan hidden→output. Latihan buku memprediksi token terakhir dari lima posisi sebelumnya pada 100 baris pertama bAbI yang dipadding di kiri. Quick=80 langkah, full=1.000. Padded batch adalah data buku; ketepatan di bawah hanya akurasi training karena listing juga melatih batch itu berulang.

In [9]:
from src.datasets import babi
tokens=babi('train',1000)
assert max(map(len,tokens))<=6
tokens=[['-']*(6-len(s))+s for s in tokens]
vocab=sorted({w for s in tokens for w in s}); lookup={w:i for i,w in enumerate(vocab)}
data=np.array([[lookup[w] for w in s] for s in tokens])
np.random.seed(0)
embed=Embedding(len(vocab),16); model=RNNCell(16,16,len(vocab))
optim=SGD(embed.get_parameters()+model.get_parameters(),.05); history=[]
for step in range(1000 if FULL else 80):
    hidden=model.init_hidden(100)
    for t in range(5): output,hidden=model(embed(data[:100,t]),hidden)
    loss=output.cross_entropy(data[:100,5]); history.append(loss.data.item())
    loss.backward(); optim.step()
hidden=model.init_hidden(100)
for t in range(5): output,hidden=model(embed(data[:100,t]),hidden)
accuracy=float(np.mean(output.data.argmax(1)==data[:100,5]))
print('Loss awal/akhir:',history[0],history[-1], 'training accuracy:',accuracy)
print('Context:',[vocab[i] for i in data[0,:5]],'target:',vocab[data[0,5]],'prediksi:',vocab[output.data[0].argmax()])
assert history[-1]<history[0]
record(autograd_gradient_max_error=autograd_error,rnn_steps=len(history),rnn_train_accuracy=accuracy,rnn_final_loss=history[-1])

Loss awal/akhir: 4.833070590863469 3.373424656761985 training accuracy: 0.14
Context: ['-', 'mary', 'moved', 'to', 'the'] target: bathroom. prediksi: office.


## Analisis output aktual — run tersimpan

Mode **quick**, eksekusi 2026-10-16T01:51:14.108887+07:00. Ringkasan ini dibuat dari output validator, bukan angka contoh dalam buku.

Autograd lolos gradient check (galat 6.331e-12) dan uji multiuse/indexing. RNN dilatih 80 langkah, loss akhir 3.3734, akurasi training 14.0%; ini belum mengukur generalisasi.

## Kesimpulan dan pembahasan pertanyaan

Framework kecil menyusun ulang operasi NumPy yang sudah dipelajari. Uji multiuse, embedding berulang, gradient checking, dan kesetaraan manual menguji mekanisme inti. Kode modul src/autograd.py identik dengan implementasi lengkap yang ditampilkan di notebook ini.

**Mengapa gradien perlu dijumlahkan?** Parameter yang sama dapat memengaruhi loss lewat banyak jalur; semua jalur masuk dalam turunan total.

**Apa tugas optimizer?** Memperbarui parameter, bukan menghitung chain rule.

**Mengapa backward graph perlu urutan?** Parent harus menerima seluruh kontribusi sebelum gradiennya diteruskan.

**Apakah RNN memakai bobot baru setiap timestep?** Tidak; parameter dibagi, aktivasi dan node graph-nya berbeda.

In [10]:
finite(*[v for v in METRICS.values() if isinstance(v, (int, float))])
print('VALIDATION_METRICS=' + json.dumps(METRICS, sort_keys=True))

VALIDATION_METRICS={"autograd_gradient_max_error": 6.3307276088053754e-12, "rnn_final_loss": 3.373424656761985, "rnn_steps": 80, "rnn_train_accuracy": 0.14}
